# Avance 2 · 02 — CNN (EfficientNet-B0): foto_natural / texto_incrustado / codigo_qr

Corre en **VS Code local (CUDA en WSL2 / MPS en Mac)** o **Colab con GPU**. La CNN detecta el *vector* (hay texto o QR en la imagen); la *intención* la decide OCR/QR → clasificador de texto (notebook 03).


In [ ]:
import os, sys
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    !git clone -b avance-2 https://github.com/bran-dort-99/prompt-security-assistant.git
    %cd prompt-security-assistant
    %pip -q install -r requirements-ml.txt
else:
    while not os.path.isdir('src') and os.getcwd() != '/':
        os.chdir('..')
    assert os.path.isdir('src'), 'Abre el notebook dentro del repo'
    %pip -q install pillow 'qrcode[pil]' matplotlib scikit-learn pandas torch torchvision
import torch
from src.vision.modelo import dispositivo
print(os.getcwd(), '| dispositivo:', dispositivo(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Fotos base (COCO val2017) y generación del dataset sintético

In [ ]:
import pathlib, subprocess
if not pathlib.Path('data/coco/val2017').exists():
    pathlib.Path('data/coco').mkdir(parents=True, exist_ok=True)
    !curl -L -o data/coco/val2017.zip http://images.cocodataset.org/zips/val2017.zip
    !unzip -q data/coco/val2017.zip -d data/coco
!ls data/coco/val2017 | head -3

In [ ]:
# (requiere data/intenciones/dataset.csv del notebook 01; si no existe usa textos base)
!python -m src.vision.generar_imagenes --fondos data/coco/val2017 --salida data/imagenes
import pandas as pd
m = pd.read_csv('data/imagenes/manifest.csv')
m.groupby(['split','clase']).size().unstack()

In [ ]:
# Vista previa
from PIL import Image
import matplotlib.pyplot as plt
fig, ax = plt.subplots(3, 5, figsize=(15, 9))
for i, c in enumerate(['foto_natural', 'texto_incrustado', 'codigo_qr']):
    for j, f in enumerate(m[(m.clase == c) & (m.split == 'train')].archivo.sample(5, random_state=1)):
        ax[i, j].imshow(Image.open('data/imagenes/' + f)); ax[i, j].axis('off'); ax[i, j].set_title(c, fontsize=8)
plt.tight_layout()

## 2. Entrenamiento (2 etapas) y evaluación

Ponga las fotos reales en `data/imagenes_reales/` con su `manifest.csv` (`archivo,clase,...`) antes de correr; si no existe, solo se evalúa el test sintético.

In [ ]:
!python -m src.vision.entrenar --datos data/imagenes --reales data/imagenes_reales --ep1 3 --ep2 5 --batch 32 --workers 4

In [ ]:
import json
from IPython.display import Image as I, display
r = json.load(open('evidencias/avance2/cnn_metricas.json'))
print('dispositivo:', r['dispositivo'], '| segundos:', r['segundos_totales'])
print('sintético  acc %.3f  F1 macro %.3f' % (r['test_sintetico']['accuracy'], r['test_sintetico']['f1_macro']))
if 'test_real' in r:
    print('real (n=%d) acc %.3f  F1 macro %.3f' % (r['test_real']['n'], r['test_real']['accuracy'], r['test_real']['f1_macro']))
    display(pd.DataFrame(r['test_real']['errores']))
for f in ['cnn_curvas', 'cnn_matriz_sintetico', 'cnn_matriz_real']:
    try: display(I(f'evidencias/avance2/{f}.png'))
    except Exception: pass

## 3. Artefactos

`models/cnn_efficientnet_b0.pt` (~16 MB) y `evidencias/avance2/cnn_*` se versionan (`git add`). Imágenes generadas y COCO **no**.

In [ ]:
!ls -la models/cnn_efficientnet_b0.pt
if EN_COLAB:
    !zip -qr artefactos_cnn.zip models/cnn_efficientnet_b0.pt evidencias/avance2/cnn_*
    from google.colab import files
    files.download('artefactos_cnn.zip')